<a href="https://www.kaggle.com/code/vuhuycong/ensemble-translation?scriptVersionId=288112516" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
! pip -q install torchtext
! pip -q install pyvi
! pip -q install sacrebleu
import nltk
nltk.download('wordnet')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 3.9 MB/s eta 0:00:00


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
import math
import json
from pyvi import ViTokenizer
from nltk import wordpunct_tokenize
import nltk
import sacrebleu 
import sentencepiece as spm
from tqdm import tqdm


# Data Preparation (BKA)



## Preprocessing data

In [3]:
def tokenize_vietnamese(text):
    """
    Tokenize Vietnamese text using PyVi tokenizer.
    PyVi performs word segmentation for Vietnamese.
    
    Args:
        text (str): Input Vietnamese text
        
    Returns:
        list: List of tokens
    """
    # PyVi tokenizer returns text with underscores for compound words
    tokenized_text = ViTokenizer.tokenize(text)
    # Split by whitespace to get individual tokens
    tokens = tokenized_text.split()
    return tokens


def tokenize_english(text):
    """
    Tokenize English text using NLTK wordpunct_tokenize.
    
    Args:
        text (str): Input English text
        
    Returns:
        list: List of tokens
    """
    return wordpunct_tokenize(text)


def tokenize_text(text, language='vi'):
    """
    Unified tokenization function for both languages.
    
    Args:
        text (str): Input text
        language (str): 'vi' for Vietnamese, 'en' for English
        
    Returns:
        list: List of tokens
    """
    if language == 'vi':
        return tokenize_vietnamese(text)
    elif language == 'en':
        return tokenize_english(text)
    else:
        raise ValueError(f"Unsupported language: {language}")

In [4]:
import re
import html
import unicodedata

def clean_text(text):
    """
    Applies some pre-processing on the given text.

    Steps :
    - Removing HTML tags
    - Removing punctuation
    - Lowering text
    """
    if text is None:
        return ""

    s = str(text)

    # 1) Bỏ mã hoá HTML (vd: &amp; -> &), rồi bỏ thẻ HTML
    s = html.unescape(s)
    s = re.sub(r"<[^>]+>", " ", s)

    # 2) Bỏ dấu câu (Unicode), giữ lại chữ, số, khoảng trắng
    s = "".join(ch for ch in s if not unicodedata.category(ch).startswith("P"))

    # 3) Đưa về chữ thường
    s = s.lower()

    # Chuẩn hoá khoảng trắng
    s = re.sub(r"\s+", " ", s).strip()

    return s

In [5]:
def preprocess_vietnamese(text):
    """Sử dụng PyVi để word segmentation cho tiếng Việt"""
    text = clean_text(text)
    # PyVi tách từ tiếng Việt (vd: "Hà Nội" -> "Hà_Nội")
    text = ViTokenizer.tokenize(text.lower())
    return text

def preprocess_english(text):
    """Preprocessing cho tiếng Anh"""
    text = clean_text(text)
    return text.lower()

In [6]:
def load_data(source_file, target_file):
  max_len = 50
  source_sents = open(source_file, "r").readlines()
  target_sents = open(target_file, "r").readlines()
  assert len(source_sents) == len(target_sents)

  source_data, target_data = [], []

  # preprocess source and target data
  # YOUR CODE HERE
  for source_sent, target_sent in zip(source_sents, target_sents):
    source_sent = preprocess_english(source_sent)
    target_sent = preprocess_vietnamese(target_sent)
    if len(source_sent.split()) <= max_len and len(target_sent.split()) <= max_len:
      source_data.append(source_sent)
      target_data.append(target_sent)
  return source_data, target_data

"""
OUT_DIR = "/content/KC4.0_MultilingualNMT/data/iwslt_en_vi"
train_src, train_trg = load_data(OUT_DIR+"/train.en", OUT_DIR+"/train.vi")
valid_src, valid_trg = load_data(OUT_DIR+"/tst2012.en", OUT_DIR+"/tst2012.vi")
test_src, test_trg = load_data(OUT_DIR+"/tst2013.en", OUT_DIR+"/tst2013.vi")
print(len(train_src), len(train_trg))
print(train_src[0])
print(train_trg[10])
"""

'\nOUT_DIR = "/content/KC4.0_MultilingualNMT/data/iwslt_en_vi"\ntrain_src, train_trg = load_data(OUT_DIR+"/train.en", OUT_DIR+"/train.vi")\nvalid_src, valid_trg = load_data(OUT_DIR+"/tst2012.en", OUT_DIR+"/tst2012.vi")\ntest_src, test_trg = load_data(OUT_DIR+"/tst2013.en", OUT_DIR+"/tst2013.vi")\nprint(len(train_src), len(train_trg))\nprint(train_src[0])\nprint(train_trg[10])\n'

In [7]:
"""import pandas as pd
train_df = pd.DataFrame({"source": train_src, "target": train_trg})
valid_df = pd.DataFrame({"source": valid_src, "target": valid_trg})
test_df = pd.DataFrame({"source": test_src, "target": test_trg})"""

'import pandas as pd\ntrain_df = pd.DataFrame({"source": train_src, "target": train_trg})\nvalid_df = pd.DataFrame({"source": valid_src, "target": valid_trg})\ntest_df = pd.DataFrame({"source": test_src, "target": test_trg})'

## Build Vocab

In [8]:
from nltk import wordpunct_tokenize

In [9]:
from typing import List
def build_vocab(corpus, language='vi', min_freq=2):
  """
  Build a vocabulary from a corpus of text.
  Params:

  corpus: Liss[str]: the list containing input texts
  max_vocab_size: int: the maximum size of the vocabulary. If None, the vocab will take all words

  Return:
  List[str]: the list of words in the vocabulary
  """
  vocab_freq = {}
  for text in corpus:
    tokens = tokenize_text(text, language=language)
    
    for token in tokens:
      if token not in vocab_freq:
        vocab_freq[token] = 0
      vocab_freq[token] += 1
  vocab_freq = {k: v for k, v in vocab_freq.items() if v >= 2}
  vocab = sorted(vocab_freq.items(), key= lambda items: items[1], reverse=True)
  vocab = [word for word, freq in vocab]
  return vocab

## Tokenization

In [10]:
from typing import List, Dict, Optional, Union
from nltk import wordpunct_tokenize
def remove_OOV_words(text, word2idx, language = "vi") :
  """
  Remove out-of-vocabulary words from a list of tokens.

  Parameters:
  tokens (List[str]): The list of tokens to filter.
  vocab (List[str]): The list of vocabulary words.

  Returns:
  List[str]: The new list of tokens with OOV tokens replaced by '<UNK>' token and all stopwords removed.
  if the new list is empty, return None
  """
  tokens = tokenize_text(text, language)
  new_tokens = []
  for token in tokens:
    if token in word2idx.keys():
      new_tokens.append(token)
    else:
      new_tokens.append("<unk>")
  if set(new_tokens) == set(["<unk>"]):
    return None
  return new_tokens

## Dataset

In [11]:
from typing import List, Dict
from nltk import wordpunct_tokenize
def batch_tokenize(token_list: List[List[str]],
                    word2idx: Dict[str, int],
                    max_length: int=None,
                    padding: bool=True) -> List[List[int]]:
  """
  add <sos> and <eos> token to the beginning and the end respectively, then tokenize a list of sentences and pad or truncate them to a fixed length.
  Params:
  token_list: List[List[str]]: the list of sentences to tokenize
  max_length: int: the maximum length of the tokenized sentences. If None, the length will be the longest sentence in the list
  padding: bool: if True, pad the sentences to the maximum length

  Return:
  List[List[int]]: the list of tokenized sentences
  """
  if max_length is None:
    max_length = max([len(tokens) for tokens in token_list])
  token_list = [tokens[:max_length] for tokens in token_list]
  tokenized_list = []
  for tokens in token_list:
    tokenized_tokens = [word2idx["<sos>"]] + [word2idx[token] for token in tokens] + [word2idx["<eos>"]]
    if padding:
       tokenized_tokens = tokenized_tokens + [word2idx["<pad>"]] * (max_length - len(tokens))
    tokenized_list.append(tokenized_tokens)
  return tokenized_list

In [12]:
def prepare_data_with_sp(src_texts, trg_texts, src_sp, trg_sp, max_len=160):
    """
    Chuẩn bị data với SentencePiece tokenization
    """
    src_encoded = []
    trg_encoded = []
    
    for src, trg in zip(src_texts, trg_texts):
        # Encode với BOS và EOS
        src_ids = src_sp.encode(src, add_bos=True, add_eos=True)
        trg_ids = trg_sp.encode(trg, add_bos=True, add_eos=True)
        
        # Lọc câu quá dài
        if len(src_ids) <= max_len and len(trg_ids) <= max_len:
            src_encoded.append(src_ids)
            trg_encoded.append(trg_ids)
    
    return src_encoded, trg_encoded

In [13]:
from torch.utils.data import Dataset

class NMTDataset(Dataset):
    def __init__(self, data, targets):
        self.data = data
        self.targets = targets
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx], self.targets[idx]

In [14]:
import torch
def make_collate_fn(src_word2idx, trg_word2idx):
    def collate_fn(batch):
        inputs = [item[0] for item in batch]
        targets = [item[1] for item in batch]
        inputs = batch_tokenize(inputs, src_word2idx, padding=True)
        targets = batch_tokenize(targets, trg_word2idx, padding=True)
        inputs = torch.LongTensor(inputs)
        targets = torch.LongTensor(targets)
        return inputs, targets
    return collate_fn

In [15]:
def collate_fn_sp(batch, src_pad_id, trg_pad_id):
    """Collate function với padding cho SentencePiece"""
    src_batch, trg_batch = zip(*batch)
    
    # Tìm max length trong batch
    src_max_len = max(len(s) for s in src_batch)
    trg_max_len = max(len(t) for t in trg_batch)
    
    # Padding
    src_padded = []
    trg_padded = []
    
    for src, trg in zip(src_batch, trg_batch):
        src_padded.append(src + [src_pad_id] * (src_max_len - len(src)))
        trg_padded.append(trg + [trg_pad_id] * (trg_max_len - len(trg)))
    
    return (torch.LongTensor(src_padded), 
            torch.LongTensor(trg_padded))

## Embedding Layer, Positional Encoder, Self Attention

In [16]:
# Code
class Embedding(nn.Module):
  def __init__(self,vocab_size,d_model):
    super().__init__()
    self.embed = nn.Embedding(vocab_size,d_model)

  def forward(self,x):
    return self.embed(x)


In [17]:
class PositionalEncoder(nn.Module):
  def __init__(self, d_model, max_seq_len = 200, dropout = 0.1):
    super().__init__()

    self.d_model = d_model
    self.dropout = nn.Dropout(dropout)

    pe = torch.zeros(max_seq_len, d_model)
    for pos in range(max_seq_len):
      for i in range(0,d_model,2):
        pe[pos,i] = math.sin(pos/(10000**(i/d_model)))
        pe[pos,i+1] = math.cos(pos/(10000**(i/d_model)))
    pe = pe.unsqueeze(0)
    self.register_buffer('pe',pe)

  def forward(self,x):
    x = x * math.sqrt(self.d_model)
    seq_len = x.size(1)
    pe = Variable(self.pe[:,:seq_len], requires_grad=False)
    if x.is_cuda:
      pe.cuda()
    x = x + pe
    x = self.dropout(x)
    return x

In [18]:
def attention(q,k,v,mask=None,dropout=None):
  d_k = q.size(-1)
  scores = torch.matmul(q,k.transpose(-2,-1))/math.sqrt(d_k)
  if mask is not None:
    scores = scores.masked_fill(mask==0,-1e9)

  scores = F.softmax(scores,dim=-1)
  if dropout is not None:
    scores = dropout(scores)

  output = torch.matmul(scores,v)
  return output,scores

# Transformer Encoder


In [19]:
# Code
class MultiHeadAttention(nn.Module):
  def __init__(self,heads,d_model,dropout=0.1):
    super().__init__()
    assert d_model % heads == 0, "d_model must be divisible by heads"

    self.d_model = d_model
    self.d_k = d_model//heads
    self.h = heads
    self.attn = None

    self.q_linear = nn.Linear(d_model,d_model)
    self.v_linear = nn.Linear(d_model,d_model)
    self.k_linear = nn.Linear(d_model,d_model)

    self.dropout = nn.Dropout(dropout)
    self.out = nn.Linear(d_model,d_model)

  def forward(self,q,k,v,mask=None):

    bs = q.size(0)

    k = self.k_linear(k).view(bs,-1,self.h,self.d_k)
    q = self.q_linear(q).view(bs,-1,self.h,self.d_k)
    v = self.v_linear(v).view(bs,-1,self.h,self.d_k)

    k = k.transpose(1,2)
    q = q.transpose(1,2)
    v = v.transpose(1,2)

    if mask is not None:
      mask = mask.unsqueeze(1)

    scores,self.attn = attention(q=q,k=k,v=v,mask=mask,dropout=self.dropout)
    output = self.out(scores.transpose(1,2).contiguous().view(bs,-1,self.d_model))

    return output



In [20]:
class Norm(nn.Module):
  def __init__(self,d_model,eps=1e-6):
    super().__init__()

    self.size = d_model

    self.alpha = nn.Parameter(torch.ones(self.size))
    self.bias = nn.Parameter(torch.zeros(self.size))

    self.eps = eps

  def forward(self,x):
    norm = self.alpha * (x - x.mean(dim=-1,keepdim=True)) / (x.std(dim=-1,keepdim=True) + self.eps) + self.bias
    return norm

In [21]:
class ResidualConnection(torch.nn.Module):

    def __init__(self, size, dropout):
        super(ResidualConnection, self).__init__()
        self.norm = Norm(size)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, sublayer):
        sublayer_output = sublayer(self.norm(x))
        sublayer_output = self.dropout(sublayer_output)
        return x + sublayer_output

In [22]:
# Code
class FeedForward(nn.Module):
  def __init__(self,d_model,d_ff=2048,dropout=0.1):
    super().__init__()
    self.linear_1 = nn.Linear(d_model,d_ff)
    self.dropout = nn.Dropout(dropout)
    self.linear_2 = nn.Linear(d_ff,d_model)

  def forward(self,x):
    x = self.dropout(F.relu(self.linear_1(x)))
    return self.linear_2(x)


In [23]:
class EncoderLayer(nn.Module):
  def __init__(self,d_model,heads,dropout=0.1):
    super().__init__()
    self.norm_1 = Norm(d_model)
    self.norm_2 = Norm(d_model)

    self.attn = MultiHeadAttention(heads=heads,d_model=d_model,dropout=dropout)

    self.ff = FeedForward(d_model=d_model,dropout=dropout)
    self.dropout_1 = nn.Dropout(dropout)
    self.dropout_2 = nn.Dropout(dropout)

  def forward(self,x,mask):
    x2 = self.norm_1(x)
    x = x + self.dropout_1(self.attn(x2,x2,x2,mask))

    x2 = self.norm_2(x)
    x = x + self.dropout_2(self.ff(x2))

    return x

In [24]:
import copy

def get_clones(module,N):
  return nn.ModuleList([copy.deepcopy(module) for i in range(N)])

class Encoder(nn.Module):
  def __init__(self,vocab_size,d_model,N,heads,dropout):
    super().__init__()
    self.N = N

    self.embed = Embedding(vocab_size,d_model)

    self.pe = PositionalEncoder(d_model,dropout=dropout)

    self.layers = get_clones(EncoderLayer(d_model,heads,dropout),N)
    self.norm = Norm(d_model)

  def forward(self,src,mask):
    x = self.embed(src)
    x = self.pe(x)
    for i in range(self.N):
      x = self.layers[i](x,mask)
    return self.norm(x)


# Transformer Decoder

In [25]:
#Code
class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.norm_1 = Norm(d_model)
        self.norm_2 = Norm(d_model)
        self.norm_3 = Norm(d_model)

        self.attn_1 = MultiHeadAttention(heads=heads, d_model=d_model, dropout=dropout)
        self.attn_2 = MultiHeadAttention(heads=heads, d_model=d_model, dropout=dropout)

        self.ff = FeedForward(d_model=d_model, dropout=dropout)

        self.dropout_1 = nn.Dropout(dropout)
        self.dropout_2 = nn.Dropout(dropout)
        self.dropout_3 = nn.Dropout(dropout)

    def forward(self, x, encoder_output, src_mask, trg_mask):
        # Self attention
        x2 = self.norm_1(x)
        x = x + self.dropout_1(self.attn_1(x2, x2, x2, trg_mask))

        # Multi attention
        x2 = self.norm_2(x)
        x = x + self.dropout_2(self.attn_2(x2, encoder_output, encoder_output, src_mask))

        # Feed forward
        x2 = self.norm_3(x)
        x = x + self.dropout_3(self.ff(x2))

        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, N, heads, dropout):
        super().__init__()
        self.N = N

        self.embed = nn.Embedding(vocab_size, d_model)
        self.pe = PositionalEncoder(d_model, dropout=dropout)

        self.layers = get_clones(DecoderLayer(d_model, heads, dropout), N)
        self.norm = Norm(d_model)

    def forward(self, trg, encoder_output, src_mask, trg_mask):
        x = self.embed(trg)
        x = self.pe(x)

        for i in range(self.N):
            x = self.layers[i](x, encoder_output, src_mask, trg_mask)

        return self.norm(x)

In [26]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, trg_vocab, d_model, N, heads, dropout):
        super().__init__()
        self.encoder = Encoder(src_vocab, d_model, N, heads, dropout)
        self.decoder = Decoder(trg_vocab, d_model, N, heads, dropout)
        self.out = nn.Linear(d_model, trg_vocab)

    def forward(self, src, trg, src_mask=None, trg_mask=None):

        encoder_output = self.encoder(src, src_mask)

        decoder_output = self.decoder(trg, encoder_output, src_mask, trg_mask)

        output = self.out(decoder_output)

        return output

### Mask

In [27]:
def nopeak_mask(size, device):
  """Tạo mask cho decoder."""
  np_mask = np.triu(np.ones((1, size, size)),k=1).astype('uint8')
  np_mask = Variable(torch.from_numpy(np_mask) == 0).to(device)
  return np_mask

def create_masks(src, trg, src_pad, trg_pad, device):
  """Tạo mask cho encoder và decoder."""
  src_mask = (src != src_pad).unsqueeze(-2)

  if trg is not None:
    trg_mask = (trg != trg_pad).unsqueeze(-2)
    size = trg.size(1)
    np_mask = nopeak_mask(size, device)
    if trg.is_cuda:
      np_mask.cuda()
    trg_mask = trg_mask & np_mask

  else:
    trg_mask = None
  return src_mask, trg_mask

In [28]:
from nltk.corpus import wordnet
import re

def get_synonym(word, SRC):
 syns = wordnet.synsets(word)
 for s in syns:
  for l in s.lemmas():
    if SRC.vocab.stoi[l.name()] != 0:
      return SRC.vocab.stoi[l.name()]

 return 0

def multiple_replace(dict, text):
  regex = re.compile("(%s)" % "|".join(map(re.escape, dict.keys())))
  return regex.sub(lambda mo: dict[mo.string[mo.start():mo.end()]], text)

## Beam Search Algorithm

In [29]:
def step(model, optimizer, src, trg, criterion, device, pad):
    """Training step."""
    model.train()

    trg_input = trg[:, :-1]
    src_mask, trg_mask = create_masks(src, trg_input, pad, trg_pad, device)

    preds = model(src, trg_input, src_mask, trg_mask)
    ys = trg[:, 1:].contiguous().view(-1)

    optimizer.zero_grad()
    loss = criterion(preds.view(-1, preds.size(-1)), ys)
    loss.backward()


    optimizer.step_and_update_lr()

    return loss.item()

In [30]:


def validate(model, valid_dataloader, criterion, device,pad):
    """Validation function."""
    model.eval()
    src_list = []
    trg_list = []
    total_loss = []
    with torch.no_grad():
        total_loss = []
        for src, trg in valid_dataloader:
            src_list.append(src)
            trg_list.append(trg)
            src = src.to(device)
            trg = trg.to(device)
            trg_input = trg[:, :-1]

            src_mask, trg_mask = create_masks(src, trg_input, pad, pad, device)
            preds = model(src, trg_input, src_mask, trg_mask)

            ys = trg[:, 1:].contiguous().view(-1)

            loss = criterion(preds.view(-1, preds.size(-1)), ys)
            total_loss.append(loss.item())

    avg_loss = np.mean(total_loss)
    perplexity = np.exp(avg_loss)
    return avg_loss, perplexity

## Optimizer

In [31]:
# Code
class ScheduleOptim():
  def __init__(self, optimizer,init_lr, d_model, n_warmup_steps):
    self.optimizer = optimizer
    self.init_lr = init_lr
    self.d_model = d_model
    self.n_warmup_steps = n_warmup_steps
    self.n_steps = 0

  def step_and_update_lr(self):
    self._update_learning_rate()
    self.optimizer.step()

  def zero_grad(self):
    self.optimizer.zero_grad()

  def _get_lr_scale(self):
    d_model = self.d_model
    n_steps, n_warmup_steps = self.n_steps, self.n_warmup_steps
    return (d_model ** -0.5) * min(n_steps ** (-0.5), n_steps * n_warmup_steps ** (-1.5))

  def state_dict(self):
    optimizer_state_dict = {
        'init_lr': self.init_lr,
        'd_model': self.d_model,
        'n_warmup_steps': self.n_warmup_steps,
        'n_steps': self.n_steps,
        'optimizer': self.optimizer.state_dict()
    }
    return  optimizer_state_dict

  def load_state_dict(self, state_dict):
    self.init_lr = state_dict['init_lr']
    self.d_model = state_dict['d_model']
    self.n_warmup_steps = state_dict['n_warmup_steps']
    self.n_steps = state_dict['n_steps']
    self.optimizer.load_state_dict(state_dict["optimizer"])

  def _update_learning_rate(self):
    self.n_steps += 1
    lr = self.init_lr + self._get_lr_scale()

    for param_group in self.optimizer.param_groups:
      param_group["lr"] = lr

In [32]:
def ensemble_translate_sp(models, text, src_sp, trg_sp, device, 
                         max_len=160, beam_size=5, ensemble_method='average'):
    """
    Ensemble translation với nhiều models và SentencePiece
    
    Args:
        models: List các model đã train
        text: Câu cần dịch
        src_sp, trg_sp: SentencePiece processors
        ensemble_method: 'average' hoặc 'voting'
    """
    for model in models:
        model.eval()
    
    # Preprocess
    text = preprocess_english(text)
    src_ids = src_sp.encode(text, add_bos=True, add_eos=True)
    src_tensor = torch.LongTensor([src_ids]).to(device)
    src_mask = (src_tensor != src_sp.pad_id()).unsqueeze(-2)
    
    # Encode với tất cả models
    encoder_outputs = []
    with torch.no_grad():
        for model in models:
            enc_out = model.encoder(src_tensor, src_mask)
            encoder_outputs.append(enc_out)
    
    # Beam search với ensemble
    output = beam_search_decode_ensemble_sp(
        models, src_tensor, src_mask, encoder_outputs,
        max_len, trg_sp.bos_id(), trg_sp.eos_id(),
        src_sp.pad_id(), device, beam_size, ensemble_method
    )
    
    # Decode
    output_ids = output[0].cpu().tolist()
    output_ids = [id for id in output_ids 
                  if id not in [trg_sp.bos_id(), trg_sp.eos_id(), trg_sp.pad_id()]]
    
    translation = trg_sp.decode(output_ids)
    return translation


def beam_search_decode_ensemble_sp(models, src, src_mask, encoder_outputs,
                                   max_len, bos_id, eos_id, pad_id,
                                   device, beam_size=5, ensemble_method='average'):
    """Beam search với ensemble của nhiều models"""
    weights = [0.5, 0.3, 0.2]
    def nopeak_mask(size):
        np_mask = np.triu(np.ones((1, size, size)), k=1).astype('uint8')
        return (torch.from_numpy(np_mask) == 0).to(device)
    
    beams = [(torch.LongTensor([[bos_id]]).to(device), 0.0)]
    completed = []
    
    for step in range(max_len - 1):
        candidates = []
        
        for seq, score in beams:
            if seq[0, -1].item() == eos_id:
                completed.append((seq, score))
                continue
            
            trg_mask = nopeak_mask(seq.size(1))
            
            # Tính log_probs từ tất cả models
            ensemble_log_probs = []
            weighted_log_probs = []
            with torch.no_grad():
                for i, (model, weight) in enumerate(zip(models, weights)):
                    out = model.decoder(seq, encoder_outputs[i], src_mask, trg_mask)
                    logits = model.out(out[:, -1])
                    log_probs = F.log_softmax(logits, dim=-1)
                    ensemble_log_probs.append(log_probs)
                    weighted_log_probs.append(weight * log_probs)
            # Ensemble các predictions
            if ensemble_method == 'average':
                # Average log probabilities
                combined_log_probs = torch.stack(weighted_log_probs).mean(dim=0)
            elif ensemble_method == 'voting':
                # Soft voting: average probabilities then take log
                probs = [torch.exp(lp) for lp in ensemble_log_probs]
                combined_probs = torch.stack(probs).mean(dim=0)
                combined_log_probs = torch.log(combined_probs + 1e-10)
            else:
                raise ValueError(f"Unknown ensemble method: {ensemble_method}")
            
            topk_log_probs, topk_indices = torch.topk(combined_log_probs, beam_size)
            
            for k in range(beam_size):
                token_log_prob = topk_log_probs[0, k].item()
                token_idx = topk_indices[0, k].item()
                
                new_seq = torch.cat([
                    seq,
                    torch.LongTensor([[token_idx]]).to(device)
                ], dim=1)
                
                new_score = score + token_log_prob
                candidates.append((new_seq, new_score))
        
        candidates.sort(key=lambda x: x[1], reverse=True)
        beams = candidates[:beam_size]
        
        if len(completed) >= beam_size:
            break
    
    completed.extend(beams)
    
    if completed:
        completed.sort(key=lambda x: x[1] / len(x[0][0]), reverse=True)
        return completed[0][0]
    
    return beams[0][0]

In [33]:
def evaluate_bleu_ensemble_sp(models, test_data_pairs, src_sp, trg_sp, 
                              device, max_len=160, beam_size=5, 
                              ensemble_method='average'):
    """
    Evaluate BLEU score với ensemble models
    """
    import sacrebleu
    
    for model in models:
        model.eval()
    
    references = []
    hypotheses = []
    
    print(f"Translating with {len(models)} models ensemble...")
    for i, (src_tokens, trg_tokens) in enumerate(tqdm(test_data_pairs)):
        if i % 100 == 0:
            print(f"Progress: {i}/{len(test_data_pairs)}")
        
        src_text = ' '.join(src_tokens)
        trg_text = ' '.join(trg_tokens)
        
        # Ensemble translate
        pred_text = ensemble_translate_sp(
            models, src_text, src_sp, trg_sp,
            device, max_len, beam_size, ensemble_method
        )
        
        references.append(trg_text)
        hypotheses.append(pred_text)
    
    # Compute BLEU
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    
    print(f"\n{'='*70}")
    print(f"Ensemble BLEU Score: {bleu.score:.2f}")
    print(f"Signature: {bleu.format()}")
    print('='*70)
    
    return bleu, references, hypotheses

In [34]:
def evaluate_bleu_with_sp(model, test_data_pairs, src_sp, trg_sp, 
                          device, max_len=160, beam_size=5):
    """
    Evaluate BLEU score với SentencePiece
    
    Args:
        model: Translation model
        test_data_pairs: List of (src_tokens_list, trg_tokens_list) - word tokens chưa encode
        src_sp: Source SentencePiece tokenizer (hoặc shared)
        trg_sp: Target SentencePiece tokenizer (hoặc shared)
        device: torch device
        max_len: Maximum generation length
        beam_size: Beam size for beam search
    
    Returns:
        bleu_score: SacreBLEU score object
    """
    import sacrebleu
    
    model.eval()
    
    references = []
    hypotheses = []
    
    print("Translating test set...")
    for i, (src_tokens, trg_tokens) in enumerate(tqdm(test_data_pairs)):
        if i % 100 == 0:
            print(f"Progress: {i}/{len(test_data_pairs)}")
        
        # Join tokens thành text
        src_text = ' '.join(src_tokens)
        trg_text = ' '.join(trg_tokens)
        
        # Translate
        pred_text = translate_sentence_sp(
            model, src_text, src_sp, trg_sp,
            device, max_len, beam_size
        )
        
        references.append(trg_text)
        hypotheses.append(pred_text)
    print(references[:10])
    print(hypotheses[:10])
    # Compute SacreBLEU
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    
    print(f"\\n{'='*70}")
    print(f"BLEU Score: {bleu.score:.2f}")
    print(f"Signature: {bleu.format()}")
    print('='*70)
    
    return bleu

def translate_sentence_sp(model, text, src_sp, trg_sp, device, 
                         max_len=160, beam_size=5):
    """
    Dịch câu với SentencePiece tokenization
    """
    model.eval()
    
    # Preprocess (giả sử src là EN, trg là VI)
    text = preprocess_english(text)
    
    # Encode
    src_ids = src_sp.encode(text, add_bos=True, add_eos=True)
    src_tensor = torch.LongTensor([src_ids]).to(device)
    
    # Create mask
    src_mask = (src_tensor != src_sp.pad_id()).unsqueeze(-2)
    
    # Encode
    with torch.no_grad():
        encoder_output = model.encoder(src_tensor, src_mask)
    
    # Beam search decode
    output = beam_search_decode_sp(
        model, src_tensor, src_mask, encoder_output,
        max_len, trg_sp.bos_id(), trg_sp.eos_id(),
        src_sp.pad_id(), device, beam_size
    )
    
    # Decode
    output_ids = output[0].cpu().tolist()
    
    # Remove BOS/EOS/PAD
    output_ids = [id for id in output_ids 
                  if id not in [trg_sp.bos_id(), trg_sp.eos_id(), trg_sp.pad_id()]]
    
    # Decode to text
    translation = trg_sp.decode(output_ids)
    
    return translation

def beam_search_decode_sp(model, src, src_mask, encoder_output,
                          max_len, bos_id, eos_id, pad_id,
                          device, beam_size=5):
    """Beam search với SentencePiece IDs"""
    
    def nopeak_mask(size):
        np_mask = np.triu(np.ones((1, size, size)), k=1).astype('uint8')
        np_mask = torch.from_numpy(np_mask) == 0
        return np_mask.to(device)
    
    beams = [(torch.LongTensor([[bos_id]]).to(device), 0.0)]
    completed = []
    
    for step in range(max_len - 1):
        candidates = []
        
        for seq, score in beams:
            if seq[0, -1].item() == eos_id:
                completed.append((seq, score))
                continue
            
            trg_mask = nopeak_mask(seq.size(1))
            
            with torch.no_grad():
                out = model.decoder(seq, encoder_output, src_mask, trg_mask)
                logits = model.out(out[:, -1])
                log_probs = F.log_softmax(logits, dim=-1)
            
            topk_log_probs, topk_indices = torch.topk(log_probs, beam_size)
            
            for k in range(beam_size):
                token_log_prob = topk_log_probs[0, k].item()
                token_idx = topk_indices[0, k].item()
                
                new_seq = torch.cat([
                    seq,
                    torch.LongTensor([[token_idx]]).to(device)
                ], dim=1)
                
                new_score = score + token_log_prob
                candidates.append((new_seq, new_score))
        
        candidates.sort(key=lambda x: x[1], reverse=True)
        beams = candidates[:beam_size]
        
        if len(completed) >= beam_size:
            break
    
    completed.extend(beams)
    
    if completed:
        completed.sort(key=lambda x: x[1] / len(x[0][0]), reverse=True)
        return completed[0][0]
    
    return beams[0][0]

# Bleu Eval

In [35]:
class LabelSmoothingLoss(nn.Module):
  def __init__(self, classes, padding_idx, smoothing=0.0,dim = -1):
    super(LabelSmoothingLoss, self).__init__()
    self.confidence = 1.0 - smoothing
    self.smoothing = smoothing
    self.cls = classes
    self.dim = dim
    self.padding_idx = padding_idx

  def forward(self, pred, target):
    pred = pred.log_softmax(dim=self.dim)
    with torch.no_grad():
      true_dist = torch.zeros_like(pred)
      true_dist.fill_(self.smoothing / (self.cls - 2))
      true_dist.scatter_(1, target.data.unsqueeze(1),self.confidence)
      true_dist[:, self.padding_idx] = 0
      mask = torch.nonzero(target.data == self.padding_idx, as_tuple= False)
      if mask.dim() > 0:
        true_dist.index_fill_(0, mask.squeeze(), 0.0)
    return torch.mean(torch.sum(-true_dist * pred, dim=self.dim))


# Training and Bleu Eval

In [36]:
class config:
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  train_src_path = "/content/iwslt15/train.en"
  train_trg_path = "/content/iwslt15/train.vi"
  valid_src_path = "/content/iwslt15/tst2012.en"
  valid_trg_path = "/content/iwslt15/tst2012.vi"
  models_path = "/kaggle/working/"
  src_lang = "en"
  trg_lang = "vi"
  n_layers = 6
  heads = 8
  dropout = 0.1
  max_strlen = 160
  batch_size =32
  epochs = 200
  max_seq_len = 200
  d_model = 512
  printevery = 400
  patience = 5
  n_iterations = 3
class Config2():
    """Model 2: Deeper model with different dropout"""
    model_name = "model2_deeper"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_layers = 8  # Tăng layers
    heads = 8
    d_model = 512
    dropout = 0.15  # Tăng dropout
    seed = 123
    max_strlen = 160
    batch_size =32
    epochs = 10
    max_seq_len = 200
    printevery = 400
    patience = 5
    n_iterations = 3

class Config3():
    """Model 3: Wider model with different architecture"""
    model_name = "model3_wider"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_layers = 6
    heads = 16  # Tăng số heads
    d_model = 768  # Tăng d_model
    dropout = 0.1
    seed = 999
    max_strlen = 160
    batch_size =32
    epochs = 10
    max_seq_len = 200
    printevery = 400
    patience = 5
    n_iterations = 3

In [37]:
import matplotlib.pyplot as plt
def plot_losses(train_losses, val_losses,epoch,save_path="/kaggle/working/"):
        """Vẽ đồ thị loss"""
        path = os.path.join(save_path,f"Summary_{epoch}")
        plt.figure(figsize=(12, 4))

        # Loss
        plt.subplot(1, 2, 1)
        plt.plot(train_losses, label='Train Loss')
        plt.plot(val_losses, label='Val Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training and Validation Loss')
        plt.legend()
        plt.grid(True)

        # Perplexity
        plt.subplot(1, 2, 2)
        train_ppl = [np.exp(loss) for loss in train_losses]
        val_ppl = [np.exp(loss) for loss in val_losses]
        plt.plot(train_ppl, label='Train PPL')
        plt.plot(val_ppl, label='Val PPL')
        plt.xlabel('Epoch')
        plt.ylabel('Perplexity')
        plt.title('Training and Validation Perplexity')
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

        print(f"Saved plot to {save_path}")


In [38]:
import pandas as pd
# Load data
OUT_DIR = "/kaggle/input/iwslt15-englishvietnamese/IWSLT'15 en-vi"
# train_src, train_trg = load_data(OUT_DIR+"/train.en.txt", OUT_DIR+"/train.vi.txt")
# valid_src, valid_trg = load_data(OUT_DIR+"/tst2012.en.txt", OUT_DIR+"/tst2012.vi.txt")
test_src, test_trg = load_data(OUT_DIR+"/tst2013.en.txt", OUT_DIR+"/tst2013.vi.txt")
# Build vocab
sp = spm.SentencePieceProcessor()
sp.load("/kaggle/input/spm-ensemble/bpe.model")

# train_src_enc, train_trg_enc = prepare_data_with_sp(
#     train_src, train_trg, sp, sp, max_len=160
# )

# valid_src_enc, valid_trg_enc = prepare_data_with_sp(
#     valid_src, valid_trg, sp, sp, max_len=160
# )

test_src_enc, test_trg_enc = prepare_data_with_sp(
    test_src, test_trg, sp, sp, max_len=160
) 

# train_dataset = NMTDataset(train_src_enc, train_trg_enc)
# valid_dataset = NMTDataset(valid_src_enc, valid_trg_enc)

# from functools import partial
# collate_fn = partial(collate_fn_sp, 
#                     src_pad_id=sp.pad_id(), 
#                     trg_pad_id=sp.pad_id())

# train_dataloader = torch.utils.data.DataLoader(
#     train_dataset, batch_size=config.batch_size, shuffle=True,
#     collate_fn=collate_fn
# )
# valid_dataloader = torch.utils.data.DataLoader(
#     valid_dataset, batch_size=config.batch_size, shuffle=False,
#     collate_fn=collate_fn
# )
test_data_fixed = []
for i, (src, trg) in enumerate(zip(test_src, test_trg)):
    src_tokens = wordpunct_tokenize(src.lower())
    trg_tokens = wordpunct_tokenize(trg.lower())
    test_data_fixed.append((src_tokens, trg_tokens))


In [39]:
src_pad = 0
trg_pad = 0
beam_size = 5

In [40]:
model = Transformer(sp.vocab_size(), sp.vocab_size(), config.d_model, config.n_layers, config.heads, config.dropout)
for p in model.parameters():
    if p.dim() > 1:
        nn.init.xavier_uniform_(p)
model = model.to(config.device)



In [41]:
# def set_seed(seed):
#     """Set seed cho reproducibility"""
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)
#     np.random.seed(seed)
#     import random
#     random.seed(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# def train_model_with_config(config, train_dataloader, valid_dataloader, 
#                            sp, start_from_scratch=True):
#     """
#     Train model với config cụ thể
    
#     Args:
#         config: Config object (Config1, Config2, hoặc Config3)
#         train_dataloader: DataLoader cho training
#         valid_dataloader: DataLoader cho validation
#         sp: SentencePiece processor
#         start_from_scratch: True để train từ đầu, False để continue
#     """
#     print(f"\n{'='*70}")
#     print(f"Training {config.model_name}")
#     print(f"Layers: {config.n_layers}, Heads: {config.heads}, "
#           f"D_model: {config.d_model}, Dropout: {config.dropout}, Seed: {config.seed}")
#     print(f"{'='*70}\n")
    
#     # Set seed
#     set_seed(config.seed)
    
#     # Khởi tạo model
#     model = Transformer(
#         sp.vocab_size(), sp.vocab_size(), 
#         config.d_model, config.n_layers, 
#         config.heads, config.dropout
#     )
    
#     # Xavier initialization
#     for p in model.parameters():
#         if p.dim() > 1:
#             nn.init.xavier_uniform_(p)
    
#     model = model.to(config.device)
    
#     # Optimizer với learning rate schedule
#     optimizer = ScheduleOptim(
#         torch.optim.Adam(model.parameters(), betas=(0.9, 0.98), eps=1e-09),
#         0.0, config.d_model, 4000
#     )
    
#     # Loss function
#     criterion = LabelSmoothingLoss(
#         sp.vocab_size(), sp.pad_id(), 
#         smoothing=0.1, dim=-1
#     )
    
#     # Load checkpoint nếu có
#     model_path = os.path.join("/kaggle/working/", f"{config.model_name}_last.pt")
#     best_model_path = os.path.join("/kaggle/working/", f"{config.model_name}_best.pt")
    
#     if os.path.exists(model_path) and not start_from_scratch:
#         print(f"Loading checkpoint from {model_path}")
#         checkpoint = torch.load(model_path, map_location=config.device, weights_only=False)
#         model.load_state_dict(checkpoint['model_state_dict'])
#         optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
#         start_epoch = checkpoint['epoch'] + 1
#         best_val_loss = checkpoint.get('best_val_loss', float('inf'))
#         train_losses = checkpoint.get('train_losses_history', [])
#         valid_losses = checkpoint.get('valid_losses_history', [])
#         print(f"Resumed from epoch {start_epoch-1}")
#     else:
#         print("Starting training from scratch")
#         start_epoch = 1
#         best_val_loss = float('inf')
#         train_losses = []
#         valid_losses = []
    
#     # Training loop
#     early_stop_counter = 0
    
#     for epoch in range(start_epoch, config.epochs + 1):
#         print(f"\nEpoch {epoch}/{config.epochs}")
#         total_loss = 0
#         start_time = time.time()
        
#         model.train()
#         train_pbar = tqdm(train_dataloader, desc=f"Training", colour="cyan")
        
#         for i, (src, trg) in enumerate(train_pbar):
#             src = src.to(config.device)
#             trg = trg.to(config.device)
            
#             train_loss = step(model, optimizer, src, trg, criterion, 
#                             config.device, sp.pad_id())
#             total_loss += train_loss
#             avg_loss = total_loss / (i + 1)
            
#             train_pbar.set_postfix({
#                 "loss": f"{avg_loss:.4f}",
#                 "lr": f"{optimizer.optimizer.param_groups[0]['lr']:.6f}"
#             })
        
#         # Validation
#         valid_loss, valid_perplexity = validate(
#             model, valid_dataloader, criterion, 
#             config.device, sp.pad_id()
#         )
        
#         epoch_time = time.time() - start_time
#         print(f"Train Loss: {avg_loss:.4f}, Val Loss: {valid_loss:.4f}, "
#               f"Val PPL: {valid_perplexity:.4f}, Time: {epoch_time:.2f}s")
        
#         train_losses.append(avg_loss)
#         valid_losses.append(valid_loss)
        
#         # Save last checkpoint
#         torch.save({
#             'epoch': epoch,
#             'model_state_dict': model.state_dict(),
#             'optimizer_state_dict': optimizer.state_dict(),
#             'val_loss': valid_loss,
#             'best_val_loss': best_val_loss,
#             'train_losses_history': train_losses,
#             'valid_losses_history': valid_losses,
#             'config': {
#                 'n_layers': config.n_layers,
#                 'heads': config.heads,
#                 'd_model': config.d_model,
#                 'dropout': config.dropout,
#                 'seed': config.seed
#             }
#         }, model_path)
        
#         # Save best model
#         if valid_loss < best_val_loss:
#             best_val_loss = valid_loss
#             early_stop_counter = 0
            
#             torch.save({
#                 'epoch': epoch,
#                 'model_state_dict': model.state_dict(),
#                 'optimizer_state_dict': optimizer.state_dict(),
#                 'val_loss': valid_loss,
#                 'train_losses_history': train_losses,
#                 'valid_losses_history': valid_losses,
#                 'config': {
#                     'n_layers': config.n_layers,
#                     'heads': config.heads,
#                     'd_model': config.d_model,
#                     'dropout': config.dropout,
#                     'seed': config.seed
#                 }
#             }, best_model_path)
            
#             print(f"✓ Saved best model with val_loss: {valid_loss:.4f}")
#         else:
#             early_stop_counter += 1
#             print(f"Early stopping counter: {early_stop_counter}/{config.patience}")
        
#         # Early stopping
#         if early_stop_counter >= config.patience:
#             print(f"\nEarly stopping triggered after {epoch} epochs")
#             break
        
#         # Plot losses every 5 epochs
#         if epoch % 5 == 0:
#             plot_losses(train_losses, valid_losses, epoch, 
#                        save_path=f"/kaggle/working/{config.model_name}_summary_{epoch}.png")
    
#     # Load best model
#     checkpoint = torch.load(best_model_path, map_location=config.device, weights_only=False)
#     model.load_state_dict(checkpoint['model_state_dict'])
    
#     print(f"\n{'='*70}")
#     print(f"Finished training {config.model_name}")
#     print(f"Best validation loss: {checkpoint['val_loss']:.4f}")
#     print(f"{'='*70}\n")
    
#     return model, checkpoint

In [42]:
# import time

# # Load data (dùng code từ script gốc)
# OUT_DIR = "/kaggle/input/iwslt15-englishvietnamese/IWSLT'15 en-vi"
# train_src, train_trg = load_data(OUT_DIR+"/train.en.txt", OUT_DIR+"/train.vi.txt")
# valid_src, valid_trg = load_data(OUT_DIR+"/tst2012.en.txt", OUT_DIR+"/tst2012.vi.txt")
# test_src, test_trg = load_data(OUT_DIR+"/tst2013.en.txt", OUT_DIR+"/tst2013.vi.txt")

# # Load SentencePiece
# sp = spm.SentencePieceProcessor()
# sp.load("/kaggle/input/spm-ds1/bpe.model")

# # Prepare data
# train_src_enc, train_trg_enc = prepare_data_with_sp(
#     train_src, train_trg, sp, sp, max_len=160
# )
# valid_src_enc, valid_trg_enc = prepare_data_with_sp(
#     valid_src, valid_trg, sp, sp, max_len=160
# )

# train_dataset = NMTDataset(train_src_enc, train_trg_enc)
# valid_dataset = NMTDataset(valid_src_enc, valid_trg_enc)

# from functools import partial
# collate_fn = partial(collate_fn_sp, 
#                     src_pad_id=sp.pad_id(), 
#                     trg_pad_id=sp.pad_id())

# # ============================================
# # TRAIN MODEL 2 (Deeper)
# # ============================================
# config2 = Config2()
# train_dataloader_2 = torch.utils.data.DataLoader(
#     train_dataset, batch_size=64, 
#     shuffle=True, collate_fn=collate_fn
# )
# valid_dataloader_2 = torch.utils.data.DataLoader(
#     valid_dataset, batch_size=config2.batch_size, 
#     shuffle=False, collate_fn=collate_fn
# )

# model2, checkpoint2 = train_model_with_config(
#     config2, train_dataloader_2, valid_dataloader_2, sp,
#     start_from_scratch=True
# )

# # ============================================
# # TRAIN MODEL 3 (Wider)
# # ============================================
# config3 = Config3()
# train_dataloader_3 = torch.utils.data.DataLoader(
#     train_dataset, batch_size=config3.batch_size, 
#     shuffle=True, collate_fn=collate_fn
# )
# valid_dataloader_3 = torch.utils.data.DataLoader(
#     valid_dataset, batch_size=config3.batch_size, 
#     shuffle=False, collate_fn=collate_fn
# )

# model3, checkpoint3 = train_model_with_config(
#     config3, train_dataloader_3, valid_dataloader_3, sp,
#     start_from_scratch=True
# )

# Bleu Eval

In [43]:
sp = spm.SentencePieceProcessor()
sp.load("/kaggle/input/spm-ensemble/bpe.model")
model_paths = [
    "/kaggle/input/en-vi-pyvi-spm/pytorch/default/1/best_model_pyvi_spm.pt",
    "/kaggle/input/ensemble-vuhhuycong/model2_deeper_best.pt", 
    "/kaggle/input/ensemble-vuhhuycong/model3_wider_best.pt"
]
config2 = Config2()
config3 = Config3()
# Khởi tạo models
ensemble_models = []

model = Transformer(
    sp.vocab_size(), sp.vocab_size(), 
    config.d_model, config.n_layers, 
    config.heads, config.dropout
)

checkpoint = torch.load(model_paths[0], map_location=config.device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(config.device)
model.eval()

ensemble_models.append(model)

model = Transformer(
    sp.vocab_size(), sp.vocab_size(), 
    config2.d_model, config2.n_layers, 
    config2.heads, config2.dropout
)

checkpoint = torch.load(model_paths[1], map_location=config2.device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(config2.device)
model.eval()

ensemble_models.append(model)

model = Transformer(
    sp.vocab_size(), sp.vocab_size(), 
    config3.d_model, config3.n_layers, 
    config3.heads, config3.dropout
)

checkpoint = torch.load(model_paths[2], map_location=config2.device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(config2.device)
model.eval()

ensemble_models.append(model)

bleu1 = evaluate_bleu_with_sp(ensemble_models[0], test_data_fixed, sp, sp, 
                              config.device, max_len=160, beam_size=5)
print(f"Model 1 BLEU: {bleu1.score:.2f}")

bleu2 = evaluate_bleu_with_sp(ensemble_models[1], test_data_fixed, sp, sp, 
                              config.device, max_len=160, beam_size=5)
print(f"Model 2 BLEU: {bleu2.score:.2f}")

bleu3 = evaluate_bleu_with_sp(ensemble_models[2], test_data_fixed, sp, sp, 
                              config.device, max_len=160, beam_size=5)
print(f"Model 3 BLEU: {bleu3.score:.2f}")
# Evaluate với ensemble
bleu_ensemble, refs, hyps = evaluate_bleu_ensemble_sp(
    ensemble_models, 
    test_data_fixed,
    sp, sp,
    config.device,
    max_len=160,
    beam_size=5,
    ensemble_method='average'  # hoặc 'voting'
)

print(f"\nEnsemble BLEU average Score: {bleu_ensemble.score:.2f}")

bleu_ensemble, refs, hyps = evaluate_bleu_ensemble_sp(
    ensemble_models, 
    test_data_fixed,
    sp, sp,
    config.device,
    max_len=160,
    beam_size=5,
    ensemble_method='voting'
)
print(f"\nEnsemble BLEU voting Score: {bleu_ensemble.score:.2f}")
# So sánh với single model
print("\nSample translations:")
for i in range(5):
    print(f"\nReference:  {refs[i]}")
    print(f"Ensemble:   {hyps[i]}")

Translating test set...


  0%|          | 0/1221 [00:00<?, ?it/s]

Progress: 0/1221


  8%|▊         | 100/1221 [02:01<13:35,  1.37it/s]

Progress: 100/1221


...

Progress: 1100/1221


 98%|█████████▊| 1200/1221 [1:16:20<01:47,  5.10s/it]

Progress: 1200/1221


100%|██████████| 1221/1221 [1:18:03<00:00,  3.84s/it]


Ensemble BLEU Score: 30.48
Signature: BLEU = 30.48 63.2/37.3/25.0/15.9 (BP = 0.980 ratio = 0.980 hyp_len = 31467 ref_len = 32118)

Ensemble BLEU voting Score: 30.48

Sample translations:

Reference:  khi tôi còn nhỏ tôi nghĩ rằng bắctriều tiên là đất_nước tốt nhất trên thế_giới và tôi thường hát bài chúng_ta chẳng có gì phải ghen_tị
Ensemble:   khi tôi còn nhỏ tôi nghĩ đất_nước của tôi là nơi tốt nhất trên hành_tinh và tôi lớn lên hát một bài hát không có gì cả

Reference:  tôi đã rất tự_hào về đất_nước tôi
Ensemble:   và tôi rất tự_hào

Reference:  ở trường chúng_tôi dành rất nhiều thời_gian để học về cuộc_đời của chủ_tịch kim ii sung nhưng lại không học nhiều về thế_giới bên ngoài ngoại_trừ việc hoa kỳ hàn_quốc và nhật bản là kẻ_thù của chúng_tôi
Ensemble:   ở trường_học chúng_tôi đã dành rất nhiều thời_gian nghiên_cứu lịch_sử kim_loại nhưng chúng_tôi chưa bao_giờ học được nhiều về thế_giới ngoại_trừ nam hàn_quốc nhật bản là kẻ_thù

Reference:  mặc_dù tôi đã từng tự hỏi không biết t

# Data Augmentation (Back Translation)